<a href="https://colab.research.google.com/github/Anast2007/Assignment-AIML/blob/main/multihomePrices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
path = kagglehub.dataset_download("camnugent/california-housing-prices")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Generate synthetic data for demonstration
np.random.seed(42)

num_samples = 1000

data = {
    'Feature_A': np.random.rand(num_samples) * 100, # Numerical feature
    'Feature_B': np.random.randint(0, 50, num_samples), # Numerical feature
    'Feature_C': np.random.choice(['X', 'Y', 'Z'], num_samples), # Categorical feature
    'Feature_D': np.random.normal(50, 10, num_samples) # Numerical feature
}

df = pd.DataFrame(data)

# Create a continuous target variable for regression tasks
df['Target_Regression'] = 2 * df['Feature_A'] + 0.5 * df['Feature_B'] - 3 * df['Feature_D'] + np.random.normal(0, 10, num_samples) + 20

# Create a binary target variable for classification tasks
# We'll make it dependent on Feature_A and Feature_B for some correlation
df['Target_Classification'] = (df['Feature_A'] + df['Feature_B'] + np.random.normal(0, 20, num_samples) > 100).astype(int)

display(df.head())
print(df.info())

Now that we have our synthetic dataset, we need to split it into training and testing sets. We will also define preprocessing steps for numerical and categorical features.

In [ ]:
# Define features (X) and target (y)
X = df.drop(columns=['Target_Regression', 'Target_Classification'])
y_reg = df['Target_Regression']
y_clf = df['Target_Classification']

# Split data into training and testing sets for regression
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X, y_reg, test_size=0.2, random_state=42)

# Split data into training and testing sets for classification
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X, y_clf, test_size=0.2, random_state=42, stratify=y_clf) # Stratify for classification


# Define preprocessing for numerical and categorical features
numerical_features = X.select_dtypes(include=np.number).columns
categorical_features = X.select_dtypes(include='object').columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

print(f"Shape of X_train_reg: {X_train_reg.shape}")
print(f"Shape of y_train_reg: {y_train_reg.shape}")
print(f"Shape of X_train_clf: {X_train_clf.shape}")
print(f"Shape of y_train_clf: {y_train_clf.shape}")

With the data prepared, we can now proceed to train and evaluate each of the requested models: Linear Regression, Logistic Regression, Decision Trees, and Random Forest.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Get the Random Forest Classifier from the pipeline
rf_classifier = random_forest_pipeline.named_steps['classifier']

# Get feature importances
feature_importances = rf_classifier.feature_importances_

# Get feature names after preprocessing
# Numerical features directly from X
numerical_feature_names = numerical_features.tolist()

# Categorical features need to be expanded due to OneHotEncoder
categorical_encoder = preprocessor.named_transformers_['cat']
categorical_feature_names_encoded = categorical_encoder.get_feature_names_out(categorical_features)

# Combine all feature names
all_feature_names = numerical_feature_names + categorical_feature_names_encoded.tolist()

# Create a DataFrame for better visualization
importance_df = pd.DataFrame({'Feature': all_feature_names, 'Importance': feature_importances})

# Sort by importance
importance_df = importance_df.sort_values(by='Importance', ascending=False)

# Plotting the feature importances
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=importance_df, palette='viridis', hue='Feature', legend=False)
plt.title('Random Forest Feature Importances')
plt.xlabel('Importance Score')
plt.ylabel('Feature Name')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("\n--- Training and Evaluating Models ---\n")

# --- Linear Regression (Regression Task) ---
print("\n### Linear Regression (Regression) ###")
linear_reg_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                      ('regressor', LinearRegression())])
linear_reg_pipeline.fit(X_train_reg, y_train_reg)
y_pred_reg_lr = linear_reg_pipeline.predict(X_test_reg)

mse_lr = mean_squared_error(y_test_reg, y_pred_reg_lr)
r2_lr = r2_score(y_test_reg, y_pred_reg_lr)

print(f"Linear Regression - Mean Squared Error: {mse_lr:.2f}")
print(f"Linear Regression - R-squared: {r2_lr:.2f}")

# --- Logistic Regression (Classification Task) ---
print("\n### Logistic Regression (Classification) ###")
logistic_reg_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                        ('classifier', LogisticRegression(random_state=42, solver='liblinear'))]) # Added solver for robustness
logistic_reg_pipeline.fit(X_train_clf, y_train_clf)
y_pred_clf_lr = logistic_reg_pipeline.predict(X_test_clf)

accuracy_lr = accuracy_score(y_test_clf, y_pred_clf_lr)
precision_lr = precision_score(y_test_clf, y_pred_clf_lr)
recall_lr = recall_score(y_test_clf, y_pred_clf_lr)
f1_lr = f1_score(y_test_clf, y_pred_clf_lr)
cm_lr = confusion_matrix(y_test_clf, y_pred_clf_lr)

print(f"Logistic Regression - Accuracy: {accuracy_lr:.2f}")
print(f"Logistic Regression - Precision: {precision_lr:.2f}")
print(f"Logistic Regression - Recall: {recall_lr:.2f}")
print(f"Logistic Regression - F1-Score: {f1_lr:.2f}")
print(f"Logistic Regression - Confusion Matrix:\n{cm_lr}")

# --- Decision Tree Classifier (Classification Task) ---
print("\n### Decision Tree Classifier (Classification) ###")
decision_tree_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                         ('classifier', DecisionTreeClassifier(random_state=42))])
decision_tree_pipeline.fit(X_train_clf, y_train_clf)
y_pred_clf_dt = decision_tree_pipeline.predict(X_test_clf)

accuracy_dt = accuracy_score(y_test_clf, y_pred_clf_dt)
precision_dt = precision_score(y_test_clf, y_pred_clf_dt)
recall_dt = recall_score(y_test_clf, y_pred_clf_dt)
f1_dt = f1_score(y_test_clf, y_pred_clf_dt)
cm_dt = confusion_matrix(y_test_clf, y_pred_clf_dt)

print(f"Decision Tree - Accuracy: {accuracy_dt:.2f}")
print(f"Decision Tree - Precision: {precision_dt:.2f}")
print(f"Decision Tree - Recall: {recall_dt:.2f}")
print(f"Decision Tree - F1-Score: {f1_dt:.2f}")
print(f"Decision Tree - Confusion Matrix:\n{cm_dt}")

# --- Random Forest Classifier (Classification Task) ---
print("\n### Random Forest Classifier (Classification) ###")
random_forest_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                         ('classifier', RandomForestClassifier(random_state=42))])
random_forest_pipeline.fit(X_train_clf, y_train_clf)
y_pred_clf_rf = random_forest_pipeline.predict(X_test_clf)

accuracy_rf = accuracy_score(y_test_clf, y_pred_clf_rf)
precision_rf = precision_score(y_test_clf, y_pred_clf_rf)
recall_rf = recall_score(y_test_clf, y_pred_clf_rf)
f1_rf = f1_score(y_test_clf, y_pred_clf_rf)
cm_rf = confusion_matrix(y_test_clf, y_pred_clf_rf)

print(f"Random Forest - Accuracy: {accuracy_rf:.2f}")
print(f"Random Forest - Precision: {precision_rf:.2f}")
print(f"Random Forest - Recall: {recall_rf:.2f}")
print(f"Random Forest - F1-Score: {f1_rf:.2f}")
print(f"Random Forest - Confusion Matrix:\n{cm_rf}")
